In [2]:
import os
import zipfile
from google.cloud import bigquery
from google.cloud import storage
from datetime import datetime
import pytz
import io

# =============================================================================
# 1. CONFIGURACIÓN DE VARIABLES
# =============================================================================

tz_lima = pytz.timezone("America/Lima")
now = datetime.now(tz_lima)
date_str = now.strftime("%Y%m%d")
ts_full  = now.strftime("%Y%m%d_%H%M%S")

PROJECT_ID = "prd-izipay-data-storage-pv"
DATASET_ID = "mc2253"
FILE_BASE_NAME = "base_comercio_bcp"
TEMP_TABLE_ID = f"TMP_{FILE_BASE_NAME}_{ts_full}"

BUCKET_NAME = "adls-reportes"
BASE_PATH = "Data/Reportes_varios"

EXPORT_PREFIX = f"{BASE_PATH}/tmp_{FILE_BASE_NAME}_{ts_full}"
FINAL_FILE_PATH = f"{BASE_PATH}/{FILE_BASE_NAME}_{date_str}.csv"
FINAL_ZIP_PATH = f"{BASE_PATH}/{FILE_BASE_NAME}_{date_str}.zip"

QUERY_SQL = """
select
  a.cod_comercio                                                                as cod_comercio,
  trim(aead.decrypt_string(e.key, a.correo_representante_legal, e.constant))   as correo_representante_legal,
  trim(aead.decrypt_string(f.key, a.correo_comercial, f.constant))             as correo_comercial,
  a.nom_comercio                                                               as nom_comercio,
  a.cod_banco_pago_comercio                                                    as cod_banco_pago_comercio,
  a.nom_banco_pago_comercio                                                    as nom_banco_pago_comercio,
  a.cod_situacion_comercio                                                     as cod_situacion_comercio,
  a.segmento_parque                                                            as segmento_parque,
  -- agrupación comercial de segmento_parque: retail queda igual, bpe pasa a negocio, be/bc/bi se agrupan como corpo, el resto (incl. null) como otros
  case
    when upper(trim(a.segmento_parque)) = 'RETAIL'          then 'RETAIL'
    when upper(trim(a.segmento_parque)) = 'BPE'              then 'NEGOCIO'
    when upper(trim(a.segmento_parque)) in ('BE', 'BC', 'BI') then 'CORPORATIVO'
    else 'OTROS'
  end                                                                          as segmento_parque_grupo
from `prd-izipay-data-storage-pv.master_party.m_comercio` a
left join `prd-izipay-data-sensitive.secure_secrets.config_protected_data` e on (1=1 and e.code = 'C_EMAIL')
left join `prd-izipay-data-sensitive.secure_secrets.config_protected_data` f on (1=1 and f.code = 'C_FULL_NAME')
where a.flag_parque = true
  and a.cod_situacion_comercio = '1'
  and a.cod_banco_pago_comercio in ('2')
"""

#`dev-izipay-data-storage.mc2253.BASE_SCORING_PERSONA_JURIDICA`
#`dev-izipay-data-storage.mc2253.BASE_SCORING_PERSONA_NATURAL`

# =============================================================================
# 2. INICIALIZACIÓN DE CLIENTES
# =============================================================================
bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

# =============================================================================
# 3. OBTENER CABECERA DESDE EL SCHEMA DE BQ
# =============================================================================
#print("Obteniendo schema de la tabla...")
#table_ref = bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TEMP_TABLE_ID}")

# =============================================================================
# 4. EJECUCIÓN Y EXPORTACIÓN
# =============================================================================
print("Paso 1: Creando tabla temporal en BigQuery...")
job_config_query = bigquery.QueryJobConfig(
    destination=f"{PROJECT_ID}.{DATASET_ID}.{TEMP_TABLE_ID}",
    write_disposition="WRITE_TRUNCATE"
)
bq_client.query(QUERY_SQL, job_config=job_config_query).result()

# ✅ Obtener cabecera desde el schema real de la tabla ya creada
table_ref = bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TEMP_TABLE_ID}")
header_line = ";".join([field.name for field in table_ref.schema]) + "\n"

print("\nPaso 2: Exportando a Cloud Storage (sin header)...")
destination_uri = f"gs://{BUCKET_NAME}/{EXPORT_PREFIX}/*.csv"
job_config_extract = bigquery.ExtractJobConfig(
    destination_format=bigquery.DestinationFormat.CSV,
    field_delimiter=";",
    print_header=False,  # ✅ Sin cabecera en los fragmentos
)
bq_client.extract_table(
    source=f"{PROJECT_ID}.{DATASET_ID}.{TEMP_TABLE_ID}",
    destination_uris=destination_uri,
    job_config=job_config_extract,
).result()

# =============================================================================
# 5. CONSOLIDACIÓN Y COMPRESIÓN
# =============================================================================
blobs = list(bucket.list_blobs(prefix=EXPORT_PREFIX))
csv_parts = sorted([blob for blob in blobs if blob.name.endswith(".csv")], key=lambda b: b.name)

if csv_parts:
    print(f"\nPaso 3: Consolidando {len(csv_parts)} archivos...")

    # ✅ Subir el header como un blob separado al inicio
    header_blob = bucket.blob(f"{EXPORT_PREFIX}_header.csv")
    header_blob.upload_from_string(header_line, content_type="text/plain")

    # Consolidar: header primero + todos los fragmentos
    all_parts = [header_blob] + csv_parts

    final_blob = bucket.blob(FINAL_FILE_PATH)

    if len(all_parts) <= 32:
        final_blob.compose(all_parts)
    else:
        lotes = [all_parts[i:i + 32] for i in range(0, len(all_parts), 32)]
        blobs_intermedios = []
        for idx, lote in enumerate(lotes):
            b_int = bucket.blob(f"{EXPORT_PREFIX}_int_{idx}.csv")
            b_int.compose(lote)
            blobs_intermedios.append(b_int)
        final_blob.compose(blobs_intermedios)
        for b in blobs_intermedios:
            b.delete()

    # Limpieza de fragmentos temporales
    header_blob.delete()
    for blob in csv_parts:
        blob.delete()

    # --- ZIP Y BORRADO DE CSV ---
    print(f"\nPaso 4: Comprimiendo a ZIP y borrando CSV...")
    local_csv = f"{FILE_BASE_NAME}_{date_str}.csv"
    local_zip = f"{FILE_BASE_NAME}_{date_str}.zip"

    final_blob.download_to_filename(local_csv)
    with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(local_csv)

    bucket.blob(FINAL_ZIP_PATH).upload_from_filename(local_zip)
    print(f"✅ ZIP creado en Storage: gs://{BUCKET_NAME}/{FINAL_ZIP_PATH}")

    final_blob.delete()
    if os.path.exists(local_csv):
        os.remove(local_csv)
    if os.path.exists(local_zip):
        os.remove(local_zip)  # ✅ También limpiar el zip local

 # ✅ Borrar tabla temporal de BigQuery
    bq_client.delete_table(f"{PROJECT_ID}.{DATASET_ID}.{TEMP_TABLE_ID}", not_found_ok=True)
    print(f"🧹 Tabla temporal {TEMP_TABLE_ID} eliminada de BigQuery.")

    print("🧹 Archivos CSV (sin zipear) eliminados con éxito.")

else:
    print("⚠️ No se encontraron datos para procesar.")

print("\n🏁 Proceso finalizado.")

Paso 1: Creando tabla temporal en BigQuery...

Paso 2: Exportando a Cloud Storage (sin header)...

Paso 3: Consolidando 153 archivos...

Paso 4: Comprimiendo a ZIP y borrando CSV...
✅ ZIP creado en Storage: gs://adls-reportes/Data/Reportes_varios/t_consumo_ibk_20260731.zip
🧹 Tabla temporal TMP_t_consumo_ibk_20260731_101151 eliminada de BigQuery.
🧹 Archivos CSV (sin zipear) eliminados con éxito.

🏁 Proceso finalizado.
